In [1]:
# ==== DO NOT MODIFY THIS CELL ====
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"

# HARD FAIL if Drive is not mounted
assert os.path.exists("/content/drive/MyDrive"), "Drive not mounted!"

# HARD FAIL if DB file missing (after first creation)
if not os.path.exists(DB_PATH):
    print("⚠️ comemo.db not found yet (first run only)")
else:
    print("✅ Using existing database:", DB_PATH)

con = duckdb.connect(DB_PATH)

# Sanity check
print(con.execute("SHOW TABLES").fetchdf())
# =================================



Mounted at /content/drive
✅ Using existing database: /content/drive/MyDrive/Capstone/comemo.db
           name
0  metadata_raw
1   reviews_raw


In [2]:
con.execute("""
SELECT
  'reviews_raw' AS table,
  COUNT(*) AS rows
FROM reviews_raw
UNION ALL
SELECT
  'metadata_raw',
  COUNT(*)
FROM metadata_raw
""").fetchdf()


,table,rows
0,reviews_raw,66033346
1,metadata_raw,7218481


In [ ]:
# Run metadata slim table SQL
with open('/content/drive/MyDrive/Capstone/Comemo-Dataset/sql/02_slim_tables.sql', 'r') as f:
    sql_query = f.read()

# Modify the SQL query to include ignore_errors=true and TRY_CAST for average_rating
# Use a more robust replacement for 'ignore_errors' by targeting a specific part within read_json
sql_query = sql_query.replace(
    "format='newline_delimited'",
    "format='newline_delimited',\n    ignore_errors=true"
)

# Replace average_rating with TRY_CAST to handle non-numeric values gracefully
sql_query = sql_query.replace(
    "average_rating",
    "TRY_CAST(average_rating AS DOUBLE) AS average_rating"
)

con.execute(sql_query)

In [3]:
con.execute("SHOW TABLES").fetchdf()



,name
0,metadata_raw
1,reviews_raw


In [ ]:
# Run reviews load SQL
with open('/content/drive/MyDrive/Capstone/Comemo-Dataset/sql/01_load_json.sql', 'r') as f:
    sql_query = f.read()

# Modify the SQL query for metadata_raw table to handle errors and casting
# First, add ignore_errors=true to the read_json for metadata.jsonl
# This requires a more targeted replacement
metadata_json_read_pattern = """FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl',
    format='newline_delimited'
)"""
metadata_json_read_replacement = """FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl',
    format='newline_delimited',
    ignore_errors=true
)"""
sql_query = sql_query.replace(metadata_json_read_pattern, metadata_json_read_replacement)

# Second, replace average_rating with TRY_CAST to handle non-numeric values gracefully
# This replacement is assumed to target the average_rating column in the metadata_raw SELECT statement.
sql_query = sql_query.replace(
    "average_rating",
    "TRY_CAST(average_rating AS DOUBLE) AS average_rating"
)

print("--- SQL Query to be executed ---")
print(sql_query)
print("--------------------------------")

con.execute(sql_query)

In [ ]:
con.execute("SHOW TABLES").fetchdf()

In [ ]:
con.close()
print("DuckDB connection closed. Tables should be persisted to /content/drive/MyDrive/Capstone/comemo.db")

In [ ]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")